# 🖼️ polars-vision: Comprehensive Demo

This notebook provides a complete demonstration of the **polars-vision** plugin - a high-performance vision/array processing plugin for Polars DataFrames.

## What is polars-vision?

polars-vision enables:
- **Lazy, zero-copy image processing** on DataFrame columns
- **Composable pipelines** that fuse multiple operations into single plugin calls
- **Dynamic parameters** using Polars expressions for per-row customization
- **Geometry operations** for contours, points, and bounding boxes
- **Seamless ML integration** with NumPy, PyTorch, and other frameworks

The plugin leverages **view-buffer**, a Rust crate providing stride-aware tensor operations with automatic kernel fusion.

---

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Basic Pipeline Operations](#2-basic-pipeline-operations)
3. [DType Promotion & Normalization](#3-dtype-promotion--normalization)
4. [Dynamic Parameters with Expressions](#4-dynamic-parameters-with-expressions)
5. [Geometry Operations](#5-geometry-operations)
6. [Lazy Pipeline Composition](#6-lazy-pipeline-composition)
7. [Multi-Output Pipelines](#7-multi-output-pipelines)
8. [ML Workflow: IoU Calculation](#8-ml-workflow-iou-calculation)
9. [Lazy Scalability Demo](#9-lazy-scalability-demo)
10. [PyTorch Integration](#10-pytorch-integration)
11. [Conclusion](#11-conclusion)


## 1. Setup & Imports

First, let's import the necessary packages and set up helper functions for displaying images.


In [ ]:
# Core imports
import io
import tempfile
from pathlib import Path

import numpy as np
import polars as pl
from PIL import Image
import matplotlib.pyplot as plt

# polars-vision imports
from polars_vision import (
    Pipeline,
    LazyPipelineExpr,
    CONTOUR_SCHEMA,
    POINT_SCHEMA,
    BBOX_SCHEMA,
)
from polars_vision.geometry.schemas import contour_from_points

# Display settings
plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['figure.dpi'] = 100

print(f"✅ Polars version: {pl.__version__}")
print("✅ polars-vision loaded successfully")


In [ ]:
# Helper functions for displaying images

def bytes_to_image(data: bytes) -> Image.Image:
    """Convert image bytes (PNG/JPEG) to PIL Image."""
    return Image.open(io.BytesIO(data))

def numpy_bytes_to_array(data: bytes, shape: tuple, dtype=np.uint8) -> np.ndarray:
    """Convert numpy-format bytes back to ndarray."""
    return np.frombuffer(data, dtype=dtype).reshape(shape)

def display_images(images: list, titles: list = None, cmap=None):
    """Display multiple images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, bytes):
            img = bytes_to_image(img)
        ax.imshow(img, cmap=cmap)
        ax.axis('off')
        if titles:
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()

def display_arrays(arrays: list, titles: list = None, cmap='viridis'):
    """Display multiple numpy arrays as heatmaps."""
    n = len(arrays)
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1:
        axes = [axes]
    
    for i, (ax, arr) in enumerate(zip(axes, arrays)):
        im = ax.imshow(arr, cmap=cmap)
        ax.axis('off')
        if titles:
            ax.set_title(titles[i])
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

print("✅ Helper functions defined")


In [ ]:
# Create sample test images for the demo

def create_test_image(width: int = 256, height: int = 256, pattern: str = "gradient") -> bytes:
    """Create a test image with various patterns."""
    if pattern == "gradient":
        # RGB gradient pattern
        r = np.linspace(0, 255, width, dtype=np.uint8)
        g = np.linspace(0, 255, height, dtype=np.uint8)
        img = np.zeros((height, width, 3), dtype=np.uint8)
        img[:, :, 0] = r[np.newaxis, :]  # Red gradient horizontal
        img[:, :, 1] = g[:, np.newaxis]  # Green gradient vertical
        img[:, :, 2] = 128  # Blue constant
    elif pattern == "checkerboard":
        block_size = 32
        img = np.zeros((height, width, 3), dtype=np.uint8)
        for i in range(0, height, block_size):
            for j in range(0, width, block_size):
                if ((i // block_size) + (j // block_size)) % 2 == 0:
                    img[i:i+block_size, j:j+block_size] = [255, 255, 255]
                else:
                    img[i:i+block_size, j:j+block_size] = [50, 50, 50]
    elif pattern == "circles":
        # Concentric circles
        y, x = np.ogrid[:height, :width]
        cx, cy = width // 2, height // 2
        r = np.sqrt((x - cx)**2 + (y - cy)**2)
        img = np.zeros((height, width, 3), dtype=np.uint8)
        img[:, :, 0] = ((np.sin(r / 10) + 1) * 127.5).astype(np.uint8)
        img[:, :, 1] = ((np.cos(r / 15) + 1) * 127.5).astype(np.uint8)
        img[:, :, 2] = 100
    else:
        # Random noise
        rng = np.random.default_rng(42)
        img = rng.integers(0, 256, (height, width, 3), dtype=np.uint8)
    
    # Convert to PNG bytes
    pil_img = Image.fromarray(img)
    buffer = io.BytesIO()
    pil_img.save(buffer, format='PNG')
    return buffer.getvalue()

# Create test images
test_images = {
    'gradient': create_test_image(256, 256, 'gradient'),
    'checkerboard': create_test_image(256, 256, 'checkerboard'),
    'circles': create_test_image(256, 256, 'circles'),
    'noise': create_test_image(256, 256, 'noise'),
}

# Display them
display_images(
    [test_images['gradient'], test_images['checkerboard'], test_images['circles']],
    ['Gradient', 'Checkerboard', 'Circles']
)
print(f"Created {len(test_images)} test images")


## 2. Basic Pipeline Operations

polars-vision uses a fluent **Pipeline** API to define image processing operations. A complete pipeline has three parts:

1. **Source**: How to interpret input data (image_bytes, blob, raw, file_path)
2. **Operations**: The transformations to apply (resize, grayscale, normalize, etc.)
3. **Sink**: The output format (numpy, torch, png, jpeg, blob)

### 2.1 Your First Pipeline

Let's create a simple pipeline that decodes an image and resizes it:


In [ ]:
# Define a simple resize pipeline
resize_pipe = (
    Pipeline()
    .source("image_bytes")           # Input is PNG/JPEG bytes
    .resize(height=128, width=128)   # Resize to 128x128
    .sink("png")                     # Output as PNG bytes
)

# Print the pipeline structure
print("Pipeline specification:")
print(resize_pipe)
print()

# Create a DataFrame with images
df = pl.DataFrame({
    "name": ["gradient", "checkerboard", "circles"],
    "image": [test_images['gradient'], test_images['checkerboard'], test_images['circles']]
})

# Apply the pipeline using .cv.pipeline()
result = df.with_columns(
    resized=pl.col("image").cv.pipeline(resize_pipe)
)

print(f"Original DataFrame schema: {df.schema}")
print(f"Result DataFrame schema: {result.schema}")
result.select("name", pl.col("image").bin.size().alias("original_size"), 
              pl.col("resized").bin.size().alias("resized_size"))


In [ ]:
# Display original vs resized images
row = result.row(0, named=True)
display_images(
    [row['image'], row['resized']],
    [f'Original (256x256)', f'Resized (128x128)']
)


### 2.2 Resize Filter Types

polars-vision supports three resize filter types:
- **nearest**: Fastest, best for pixel art or when speed is critical
- **bilinear**: Good balance of speed and quality
- **lanczos3**: Best quality, slower (default)
